In [2]:
########################## WORLD MAP WITH CIRCLE MARKER AND PERCENTAGE IPV4 #########
import pandas as pd
import folium
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.geocoders import Nominatim
import pycountry
import time

# Function to get coordinates for a given country
def get_coordinates(country_name):
    try:
        country = pycountry.countries.lookup(country_name)
        location = geolocator.geocode(country.name)
        if location:
            return pd.Series([location.latitude, location.longitude])
        else:
            return pd.Series([None, None])
    except Exception:
        return pd.Series([None, None])

# Load dataset
dataset = pd.read_csv("ip_alloc(in).csv")

# Group by country and sum IPv4 allocations
ip_pr_country = dataset.groupby("country name")["ipv4"].sum().reset_index()

# Extract the total IPv4 allocation from the 'World' row
world_total = ip_pr_country.loc[ip_pr_country['country name'] == 'World', 'ipv4'].values[0]

# Calculate the percentage for each country
ip_pr_country['ipv4_pct'] = (ip_pr_country['ipv4'] / world_total) * 100

# Exclude 'World' in country name column
ip_pr_country = ip_pr_country[ip_pr_country["country name"] != "World"]

# Initialize geolocator
geolocator = Nominatim(user_agent="ipv4_mapper")

# Geocode countries to get latitude and longitude
ip_pr_country[["Latitude", "Longitude"]] = ip_pr_country["country name"].apply(get_coordinates)

# Drop rows with missing coordinates
ip_pr_country.dropna(subset=["Latitude", "Longitude"], inplace=True)

# Create an interactive map
world_map = folium.Map(location=[20, 0], zoom_start=2)

# Add circle markers to the map
for _, row in ip_pr_country.iterrows():
    # Skip the 'World' entry
    if row['country name'] == 'World':
        continue
    # Ensure that latitude and longitude are available
    if pd.notnull(row['Latitude']) and pd.notnull(row['Longitude']):
         # Create the popup text
        popup_text = f"{row['country name']} \nIPv4: {int(row['ipv4']):,} \nShare: {row['ipv4_pct']:.2f}%"
        # Add the circle marker to the map
        folium.CircleMarker(
            location=[row["Latitude"], row["Longitude"]],
            radius=max(row["ipv4"] / 1e7, 2),  # Ensure a minimum radius
            color="blue",
            fill=True,
            fill_color="blue",
            fill_opacity=0.5,
            popup=folium.Popup(popup_text, parse_html=True)
        ).add_to(world_map)
        time.sleep(1)  # To respect Nominatim's usage policy

# Save the map to an HTML file
world_map.save("ipv4_allocation_map.html")
